In [1]:
import numpy as np

"""
Practically, channel has multiple taps resulting in ISI and making it hard to perform channel estimation 
as the channel matrix is modeled as a Toeplitz matrix (from convolutional operator). However, with domain knowledge 
that any circulant matrix can be diagonalized by the unitary DFT matrix, we can represent the channel as 
the diagonal matrix. 

[1] CPI and CPR for turning Toeplitz channal matrix into a circulant channel matrix.
Toeplitz matrix has similar structure to a circulant matrix, simple maniputlation on Toeplitz matrix can
result in a circulant matrix. To turn Toeplitz matrix, H_lin, into a circulant matrix H_circ,
We left- and right- multiply H_lin with manipulation matrices R and T, i.e., 
                    H_circ = R @ H_lin @ T, 
which by interpretation, T reseprents cyclic prefix insertion, and R represents cyclic prefix removal.

Let N' be the sequence length of transmitted signal (or number of transmitted symbols) 
and M be the number of taps of the channel.
The Toeplitz channel H_lin, will be a N'+ M - 1 by N' matrix.
-- R will have the shape of N by N'+ M - 1
-- T will have the shape of N' by N, where N is the actual number of symbols.
Note: N' = N + L where L is the CP length

[2] Diagonalize circulant matrix
As any circulant matrix can be diagonalized by unitary DFT matrix, i.e., 
                    H_circ = F.conj().T @ Lambda @ F 
where Lambda is a diagonal matrix.
From [1],           H_circ              =     R @ H_lin @ T              = F.conj().T @ Lambda @ F 
                F @ H_circ @ F.conj().T = F @ R @ H_lin @ T @ F.conj().T =              Lambda
Therefore, to obtain diagonal channel matrix, we follow this set of operations F @ R @ H_lin @ T @ F.conj().T
which is IDFT -> CPI -> Channel -> CPR -> DFT. Meaning that 
-- at the transmitter, we first perform IDFT on the symbols then add cyclic prefix. 
-- at the receiver, we first remove the cyclic prefix then perform DFT.

The interpretation of performing IDFT is to modulate each symbols with orthogonal frequencies and sum them up
(by definition)   x[n] = sum_k {X[k] * exp(2j pi n k / K)}.

Application: In OFDM system, following x -> IDFT -> CPI -> Channel -> CPR -> DFT -> y will equivalently result in
x -> Lambda -> y in frequency domain. Therefore, we can simply use x and y to estimate diagonal channel Lambda.

Note: After estimate the diagonal channel Lambda, it can also be used to compute back H_lin by 
                    Lambda = F @ H_circ @ F.conj().T = np.diag(F @ h * np.sqrt(N)).
                    
"""

def toeplitz(h, N):   # M: NumColumn
    M = h.size
    A = np.zeros([N + M - 1, N])
    for i in range(N):
        A[i:i+M, i] = h
    return A

def CPI(x, L):    #  L: CP length
    if L == 1: return np.concat([np.array([x[-1]]), x])
    else: return np.concat([x[-L:], x])

def CPI_mat(x, L):    #  L: CP length
    T = np.eye(x.size, dtype=int)
    T = np.concat([T[-L:,:], T], axis=0)
    return T

def CPR(x, N, L):
    return x[L:L+N]

def CPR_mat(x_cp, N, L):    #  L: CP length
    R = np.eye(N, dtype=int)
    z1 = np.zeros([N, L])
    z2 = np.zeros([N, x_cp.size - N - L])
    R = np.concat([z1,R,z2], axis=1)
    return R

def circulant(x):
    A = np.zeros([x.size, x.size], dtype=int)
    for i in range(x.size):
        A[:,i] = x
        x = np.concat([np.array([x[-1]]), x[:-1]])
    return A

def x_padded(x, N):
    y = np.zeros(N)
    y[:x.size] = x
    return y

## Configuration ##
N = 6       # number of symbols
M = 3       # number of channel tap
L = 2       # CP length: L >= M - 1

## Initialization ##
F = np.fft.fft(np.eye(N), norm='ortho')
x = np.random.randint(1, 10, N)
h = np.random.randint(1, 10, M)

print("Symbols:\t{}".format(x))
print("Channel:\t{}".format(h))


Symbols:	[4 8 4 2 1 5]
Channel:	[4 2 8]


In [2]:
IDFTSignal = F.conj().T @ x
T = CPI_mat(IDFTSignal, L)
TransmittedSignal = T @ IDFTSignal

H_lin = toeplitz(h, TransmittedSignal.size)
ChannelOuput = H_lin @ TransmittedSignal

R = CPR_mat(ChannelOuput, N, L)
ReceivedSignal = R @ ChannelOuput
ReceivedSymbols = F @ ReceivedSignal

print("Toeplitz Channel:\n\n{}\n\tSize:\t{}\n\n".format(H_lin, H_lin.shape))
print("Received Symbols:\n\n{}\n\tSize:\t{}".format(ReceivedSymbols,ReceivedSymbols.shape))



Toeplitz Channel:

[[4. 0. 0. 0. 0. 0. 0. 0.]
 [2. 4. 0. 0. 0. 0. 0. 0.]
 [8. 2. 4. 0. 0. 0. 0. 0.]
 [0. 8. 2. 4. 0. 0. 0. 0.]
 [0. 0. 8. 2. 4. 0. 0. 0.]
 [0. 0. 0. 8. 2. 4. 0. 0.]
 [0. 0. 0. 0. 8. 2. 4. 0.]
 [0. 0. 0. 0. 0. 8. 2. 4.]
 [0. 0. 0. 0. 0. 0. 8. 2.]
 [0. 0. 0. 0. 0. 0. 0. 8.]]
	Size:	(10, 8)


Received Symbols:

[56.-3.10862447e-15j  8.-6.92820323e+01j -4.+2.07846097e+01j
 20.+2.22044605e-15j -1.-5.19615242e+00j  5.+4.33012702e+01j]
	Size:	(6,)


In [3]:
IDFTSignal = F.conj().T @ x
H_circ = circulant(x_padded(h, N))
ReceivedSignal_02 = H_circ @ IDFTSignal
ReceivedSymbols_02 = F @ ReceivedSignal_02

print("Circulant Channel:\n\n{}\n\tSize:\t{}\n".format(H_circ,H_circ.shape))
print("Received Symbols 02:\n\n{}\n\tSize:\t{}\n".format(ReceivedSymbols_02,ReceivedSymbols_02.shape))
print("---- Difference:\t{}".format(np.sum(np.abs(ReceivedSymbols - ReceivedSymbols_02))))

Circulant Channel:

[[4 0 0 0 8 2]
 [2 4 0 0 0 8]
 [8 2 4 0 0 0]
 [0 8 2 4 0 0]
 [0 0 8 2 4 0]
 [0 0 0 8 2 4]]
	Size:	(6, 6)

Received Symbols 02:

[56.-3.10862447e-15j  8.-6.92820323e+01j -4.+2.07846097e+01j
 20.+2.22044605e-15j -1.-5.19615242e+00j  5.+4.33012702e+01j]
	Size:	(6,)

---- Difference:	0.0


In [4]:
Lambda = np.diag(np.diagonal(F @ H_circ @ F.conj().T))
ChannelOuput_03 = Lambda @ F @ IDFTSignal
ReceivedSignal_03 = F.conj().T @ ChannelOuput_03
ReceivedSymbols_03 = F @ ReceivedSignal_03

print("Lambda:\n\n{}\n\tSize:\t{}\n".format(Lambda, Lambda.shape))
print("Received Symbols 03:\n\n{}\n\tSize:\t{}\n".format(ReceivedSymbols_03, ReceivedSymbols_03.shape))
print("---- Difference:\t{}".format(np.sum(np.abs(ReceivedSymbols - ReceivedSymbols_03))))

Lambda:

[[14.+0.j          0.+0.j          0.+0.j          0.+0.j
   0.+0.j          0.+0.j        ]
 [ 0.+0.j          1.-8.66025404j  0.+0.j          0.+0.j
   0.+0.j          0.+0.j        ]
 [ 0.+0.j          0.+0.j         -1.+5.19615242j  0.+0.j
   0.+0.j          0.+0.j        ]
 [ 0.+0.j          0.+0.j          0.+0.j         10.+0.j
   0.+0.j          0.+0.j        ]
 [ 0.+0.j          0.+0.j          0.+0.j          0.+0.j
  -1.-5.19615242j  0.+0.j        ]
 [ 0.+0.j          0.+0.j          0.+0.j          0.+0.j
   0.+0.j          1.+8.66025404j]]
	Size:	(6, 6)

Received Symbols 03:

[56.-7.99360578e-15j  8.-6.92820323e+01j -4.+2.07846097e+01j
 20.+6.21724894e-15j -1.-5.19615242e+00j  5.+4.33012702e+01j]
	Size:	(6,)

---- Difference:	1.0541950417423585e-13


In [5]:
ChannelOuput_04 = Lambda @ x
ReceivedSymbol_04 = ChannelOuput_04

print("Received Symbols 04:\n\n{}\n\tSize:\t{}\n".format(ReceivedSymbol_04, ReceivedSymbol_04.shape))
print("---- Difference:\t{}".format(np.sum(np.abs(ReceivedSymbols - ReceivedSymbol_04))))

Received Symbols 04:

[56. +0.j          8.-69.2820323j  -4.+20.78460969j 20. +0.j
 -1. -5.19615242j  5.+43.30127019j]
	Size:	(6,)

---- Difference:	2.5476025846279334e-14
